# Workshop: Introducción a Generative AI en Oracle y Creación de Agentes con LangChain

Bienvenidos al workshop. En esta sesión vamos a explorar cómo usar los **servicios de IA generativa de Oracle** para resolver problemas reales y luego **crear un agente inteligente** usando **LangChain** que pueda interactuar con estos servicios.

## Objetivos de la sesión
- Conocer la oferta de **Oracle Cloud Infrastructure (OCI Generative AI)** y cómo integrarla desde Python.
- Ejecutar peticiones a modelos de lenguaje para **generar texto** de manera controlada.
- Construir un **agente con LangChain** que use herramientas (como SQL o RAG) para responder preguntas de forma autónoma.
- Aprender buenas prácticas para **orquestar flujos de trabajo** y extender capacidades de los modelos.

## Requisitos previos
- Conocimientos básicos de Python 🐍
- Tener acceso a una cuenta de **Oracle Cloud** con permisos para usar **OCI Generative AI**
- Familiaridad básica con entornos virtuales y Jupyter Notebooks.

> 💡 **Tip:** este notebook está diseñado para ser práctico y paso a paso. Podrás copiar, ejecutar y modificar el código para experimentar con los conceptos que vamos a explicar.

¡Vamos a empezar!

### A continuación... 

📰 Recopilaremos noticias sobre el paro del 16 de Septiembre ocurrido en la ciudad de Bogotá 

🤖 Consumiremos un modelo de lenguaje alojado en Oracle Cloud 

🔍 Construiremos un agente con langchain que es capaz de responder a preguntas relacionadas con el paro del 16 de septiembre 

## Notebook

### Instalación

In [18]:
!pip install -r requirements.txt

### Importación de librerías

In [19]:
import json
import re
from typing import Dict, List

import oci
import requests
from IPython.display import Markdown, display
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.chat_models import ChatOCIGenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from oci.auth.signers import get_resource_principals_signer
from oci.config import from_file
from tavily import TavilyClient

### Registro y configuración de Tavily

 Nuestro agente necesita acceso a la web para acceder a las últimas noticias del tema que le hemos indicado, para esto, es necesario configurar una herramienta que le permita a nuestro agente navegar por portales de noticias, por eso usaremos [Tavily](https://www.tavily.com/).

#### 🪪 Registro en Tavily (paso a paso)

1) Abre **https://app.tavily.com/home** y haz clic en **Sign Up**. Verifica tu correo electrónico para activar la cuenta. ![image1](./images/tavily_signup.png)
2) Inicia sesión: en la **página principal** verás tu **API Key**. Haz **Copy** para copiarla. ![image2](./images/api_key.png)


In [ ]:
# Pega la API Key de Tavily aquí
TAVILY_API_KEY = "Pega aqui tu llave de Tavily"

In [21]:
# No ajustes el codigo esta es solo para validar si la llave se configuro como una variable
try:
    assert TAVILY_API_KEY != "Pega aqui tu llave de Tavily", "Por favor, pega tu API Key de Tavily en la variable TAVILY_API_KEY"
    print("✅ API Key de Tavily configurada correctamente")
except AssertionError as e:
    print(f"❌ Error: {e}")
    print("⚠️ Debes configurar tu API Key de Tavily en la celda anterior")

✅ API Key de Tavily configurada correctamente


El siguiente codigo como se denota crea una lista con las paginas oficiales del pais para poder con tavily extraer informacion relevante de acuerdo al topico de noticia a buscar

In [22]:

DOMAINS = [
    "www.diariolibre.com", "elnuevodiario.com.do", "listindiario.com", "acento.com.do", "elnacional.com.do", "dominicantoday.com", "eldia.com.do", "deultimominuto.net","noticiassin.com"
]
@tool
def get_paro_comprehensive_news(pregunta:str="Que afectaciones tuvo Republica Dominicana con el Huracan Melisa") -> str:
    """Devuelve info/noticias de horarios, causas y afectaciones del Huracan Melisa en el Caribe."""

    client = TavilyClient(TAVILY_API_KEY)
    response = client.search(
    query=pregunta,
    #include_domains=DOMAINS,
    #topic="news",
    #days=45,
    #max_results=10
    )
    return response

## Configuración de la autenticación del SDK de OCI

Desde este notebook es necesario acceder a algunos servicios de Oracle, como el servicio de Generative AI, aunque ejecutes este notebook en cloud o de forma local, es necesario configurar las credenciales en la máquina que realiza el consumo del servicio. 

### Configuración de credenciales en Data Science

Ejecuta las siguientes celdas para la creación de la configuración en un ambiente de Data Science

In [ ]:
# crea carpeta y permisos
!mkdir -p /home/datascience/.oci

In [ ]:
# Ver tu HOME y listar (incluye ocultos)
# La carpeta donde esta el home en el servicio es /home/datascience
!echo $HOME
!ls -la $HOME | head -n 30

In [ ]:
# Se crea una carpeta .oci donde guarda la llave y el archivo de configuracion
!mkdir -p ~/.oci
!ls -la ~/.oci


Para obtener la configuración y el api key, es necesario seguir los siguientes pasos [Configurar credenciales en ambiente local](../utils/Configurar%20credenciales%20en%20ambiente%20local.md). Este tutorial explica cómo obtener la configuración, las credenciales y en el paso 5 se detalla cómo realizar la configuración en el ambiente local. 

Para realizar la configuración en Data Science y no en un ambiente local, reemplazaremos el paso 5 por el siguiente.

Ahora vamos a ubicar la llave privada que descargamos en los pasos anteriores al generar el API Key. El archivo tendrá un nombre similar a “tu_usuario-año-mes-diaTHH_MM_SS.XXX.pem”.

Este archivo debe renombrarse como **“oci_api_key.pem”** y cargarse en Data Science utilizando la opción “Upload Files”, o bien arrastrándolo directamente en el menú izquierdo del navegador.

Una vez que el archivo esté cargado, podremos proceder con la ejecución de la siguiente línea.

In [ ]:
# Este codigo de linux mueve la llave a la carpeta .oci y le cambia los permisos
!mv ~/oci_api_key.pem ~/.oci/oci_api_key.pem
!chmod 600 ~/.oci/oci_api_key.pem
!ls -la ~/.oci

A continuación, crearemos el archivo de configuración en la ruta ~/.oci/config, vamos a copiar los valores de la configuración mostrada en pantalla y a reemplazarlos en la siguiente línea.


Reemplazaremos **_ocid1.user.oc1.._** por el ocid del usuario mostrado en pantalla<br>
Reemplazaremos **_fingerprint_** por el figerprint mostrado en pantalla<br>
Reemplazaremos **_tenancy_** por el figerprint mostrado en pantalla<br>
Reemplazaremos **_region_** por el figerprint mostrado en pantalla<br>
🚨 No reemplazaremos **_key_file_** por ninguna ruta si estamos ejecutando este notebook en DataScience. Si queremos ejecutar este notebook de forma local, podemos reemplazar la ruta por ~/.oci/nombre_de_la_key.pem 

In [ ]:
%%bash
cat > ~/.oci/config <<'CFG'
[DEFAULT]
user=ocid1.user.oc1.....
fingerprint=4a:c7:6f:.....
tenancy=ocid1.tenancy.oc1......
region=region de tu tenant!
key_file=/home/datascience/.oci/oci_api_key.pem
CFG

echo "Config creado en ~/.oci/config"
cat ~/.oci/config | sed 's/fingerprint=.*/fingerprint=<oculto>/'

In [ ]:
# Quita posibles finales de línea de Windows (CRLF)
!sed -i 's/\r$//' ~/.oci/config

# mostrar
!sed -n '1,200p' ~/.oci/config

### Configuración de credenciales en un ambiente local

Para realizar la configuración en el ambiente local, siga los siguientes pasos [Configurar credenciales en ambiente local](../utils/Configurar%20credenciales%20en%20ambiente%20local.md).

In [ ]:
#carga a memoria la configuracion de OCI para enviarla al API de Generative AI
config = from_file()

In [ ]:
# Descomenta únicamente la línea que corresponda a tu región en la cual esta tu tenant
#REGION = "sa-saopaulo-1"
REGION = "us-chicago-1"
#REGION = "uk-london-1"
#REGION = "eu-frankfurt-1"
#REGION = "ap-osaka-1"
#REGION = "us-ashburn-1"

match REGION:
    case "us-chicago-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.us-chicago-1.amaaaaaask7dceyayjawvuonfkw2ua4bob4rlnnlhs522pafbglivtwlfzta"
    case "sa-saopaulo-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.sa-saopaulo-1.amaaaaaask7dceyarsn4m6k3aqvvgatida3omyprlcs3alrwcuusblru4jaa"
    case "uk-london-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.uk-london-1.amaaaaaask7dceyach2dyu6g5w5ocnvbkto2g76wxitj3rpddplsqoxqh2lq"
    case "eu-frankfurt-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.eu-frankfurt-1.amaaaaaask7dceya4tdabclcsqbc3yj2mozvvqoq5ccmliv3354hfu3mx6bq"
    case "ap-osaka-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.ap-osaka-1.amaaaaaask7dceyaei4b6vhy7rgsqzuqet4ndnnun4fhxco2tfzusslg35wa"
    case "us-ashburn-1":
        MODEL_OCID = "ocid1.generativeaimodel.oc1.iad.amaaaaaask7dceyah6tjdejjashngznsylutuhhvufukzb2g2ls54g2flsfq"


In [ ]:
# No ajustes el codigo esta es solo para validar si la region se configuro como una variable

assert REGION in ["sa-saopaulo-1", "us-chicago-1", "uk-london-1", "eu-frankfurt-1", "ap-osaka-1"], "Por favor, descomenta la línea que corresponda a tu región"

# No ajustes el codigo esta es solo para validar si la llave se configuro como una variable
try:
    assert REGION != "el codigo de la region", "Por favor, pega tu API Key de Tavily en la variable TAVILY_API_KEY"
    print("✅ Codigo de Region configurada correctamente")
except AssertionError as e:
    print(f"❌ Error: {e}")
    print("⚠️ Debes configurar el codigo de la region en la celda anterior")

In [ ]:
# Aquí debes pegar el OCID de tu compartimento
# Encuentra el OCID de tu compartimento en la consola de Oracle Cloud, en la sección de Compartments https://cloud.oracle.com/identity/compartments
COMPARTMENT_ID = "ocid1.tenancy.oc1.."

In [ ]:
SERVICE_ENDPOINT = f"https://inference.generativeai.{REGION}.oci.oraclecloud.com"


In [ ]:
if MODEL_OCID is None:
    from oci.generative_ai import GenerativeAiClient
    genai = GenerativeAiClient()
    models = genai.list_models(
        compartment_id=COMPARTMENT_ID,
        capability=["CHAT"],
        lifecycle_state="ACTIVE"
    ).data.items
    assert models, "No hay modelos CHAT visibles en el compartimento. Revisa permisos/compartimento."
    MODEL_ID = models[0].id
    print("Usando modelo:", MODEL_ID)

# === Cliente de inferencia ===
inf = oci.generative_ai_inference.GenerativeAiInferenceClient(
    config=config,
    service_endpoint=SERVICE_ENDPOINT
)

# === Prompt del usuario ===
user_input = "hola, tienes información del Huracan Melisa en Republica Dominicana?"

# --- Construcción del request ---
content = oci.generative_ai_inference.models.TextContent(text=user_input)
message = oci.generative_ai_inference.models.Message(role="USER", content=[content])

chat_request = oci.generative_ai_inference.models.GenericChatRequest(
    api_format=oci.generative_ai_inference.models.BaseChatRequest.API_FORMAT_GENERIC,
    messages=[message],
    max_tokens=600,
    temperature=1.0,
    frequency_penalty=0.0,
    presence_penalty=0.0,
    top_p=0.75,
)

chat_detail = oci.generative_ai_inference.models.ChatDetails(
    serving_mode=oci.generative_ai_inference.models.OnDemandServingMode(model_id=MODEL_OCID),
    chat_request=chat_request,
    compartment_id=COMPARTMENT_ID,
)

# === Llamada ===
resp = inf.chat(chat_detail)

# === Resultado ===
choices = resp.data.chat_response.choices
response_text = choices[0].message.content[0].text if choices else "No se generó respuesta."
print(json.dumps({"response": response_text}, indent=2, ensure_ascii=False))

## 🤖 Creación del Agente LangChain

In [ ]:
# Configura tu endpoint y compartimento
#ENDPOINT = f"https://inference.generativeai.{REGION}.oci.oraclecloud.com"

llm = ChatOCIGenAI(
  model_id=MODEL_OCID,
  service_endpoint=SERVICE_ENDPOINT,
  compartment_id=COMPARTMENT_ID,
  provider="meta",
  model_kwargs={
    "temperature": 0.3, 
    "max_tokens": 800,   
    "top_p": 0.8,  
    "frequency_penalty": 0,
    "presence_penalty": 0,
  },
  auth_type="API_KEY",
  auth_profile="DEFAULT"
)

tools = [get_paro_comprehensive_news]

react_prompt_template = """Eres un asistente especializado busqueda de informacion de noticias relevantes sobre el clima.
Responde en español y usa las herramientas disponibles cuando sea útil.

REGLAS IMPORTANTES:

Tras usar una herramienta, resume con tus palabras los hallazgos (no pegues el JSON).

Prioriza información reciente y oficial (Pricipales sitios web y paginas de Republica Dominicana).

Incluye zonas afectadas, recomendaciones y enlaces útiles cuando existan.

Si hay discrepancias entre fuentes, indícalas brevemente.

Herramientas disponibles:
{tools}

Usa EXACTAMENTE este formato:

Question: la pregunta a responder
Thought: explica qué harás
Action: una de [{tool_names}]
Action Input: el input para la acción (o "" si no aplica)
Observation: resultado de la acción
Thought: analiza y sintetiza
Final Answer: respuesta clara y útil en español 

Comienza.

Question: {input}
Thought: {agent_scratchpad}"""

prompt = PromptTemplate.from_template(react_prompt_template)
agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=6,
    stream_runnable=False,
    handle_parsing_errors=True
)
pregunta = "que afectaciones tuvo Republica Dominicana con el Huracan Melisa?"
respuesta = agent_executor.invoke({"input": pregunta})

pregunta = "¿Que organizaciones estuvieron coordinando a nivel del pais?"
respuesta = agent_executor.invoke({"input": pregunta})


In [ ]:
respuesta

In [ ]:
# 1) Toma la salida del agente y conviértela a texto de forma segura
def _to_text(x):
    if isinstance(x, dict):
        # LangChain AgentExecutor suele devolver {"output": "..."}
        return x.get("output") or json.dumps(x, ensure_ascii=False, indent=2)
    return str(x)
insumos = _to_text(respuesta)   # <--- usa la variable 'respuesta' que ya tienes del agente
# 2) Prompt de análisis/síntesis para Fiestas Patrias (Chile)
analysis_prompt = f"""
Eres un analista de la NHC (centro nacional de huracnes) especializado en **eventos generados por huracanes**.
Usa solo los datos entregados más abajo para responder en **español**, claro y útil.
**DATOS RECOLECTADOS** (pueden incluir JSON):
{insumos}
---
### TAREAS
1. Responde directamente a la pregunta: **"{pregunta}"**.
2. Extrae y organiza lo más importante (si está disponible):
   - **hora de inicio y fin**
   - **zonas afectadas**
   - **Ayudas de organizaciones**
   - **alternativas de transporte aereo y maritimo**
   - **enlaces oficiales**
3. Indica si se reportan **medidas de la nacion**
4. Si hay discrepancias entre fuentes, menciónalas brevemente.
---
### FORMATO (Markdown)
#### ## Resumen
- 3–5 líneas con lo esencial (hora, zonas críticas, recomendaciones).
#### ## Rutas y afectaciones (si hay datos)
- **Sistema/sector – Afectación**: detalle | horario | recomendaciones | enlace
## Recomendaciones (si hay datos)
- Lista de recomendaciones
#### ## Fuentes
- Lista de URLs citadas (solo si aparecen en los datos).
---
### REGLAS
- **No inventes** datos ni enlaces. Si algo no está en los datos, escribe: *No disponible*.
- Prioriza información **reciente y oficial** (medios confiables).
"""
# 3) Invoca la LLM (no streaming) y muestra en Markdown
analysis_response = llm.invoke(analysis_prompt)
display(Markdown(analysis_response.content))
